<a href="https://colab.research.google.com/github/2303A51689/Python-for-DS-1689/blob/main/new_dataset_with_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Supervised ML suite with classification reports
# Dataset: india_pesticide_toxicity_risk(_with_nulls).csv
# Tip: If auto-detection picks the wrong target, set TARGET_COLUMN manually.

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score

# Models
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# ====== CONFIG ======
DATA_PATHS = [
    "india_pesticide_toxicity_risk_reduced.csv"
]
TARGET_COLUMN = None   # e.g., "toxicity_risk". Leave None to auto-detect.
TEST_SIZE = 0.2
RANDOM_STATE = 42

# ====== LOAD ======
for p in DATA_PATHS:
    if os.path.exists(p):
        df = pd.read_csv(p)
        break
else:
    raise FileNotFoundError("Dataset not found. Please ensure the CSV is in /mnt/data/.")

# ====== TARGET AUTO-DETECTION ======
def auto_pick_target(dataframe):
    # Priority by name
    name_candidates = ["toxicity_risk","risk_label","label","target","class","toxicity","risk"]
    for c in dataframe.columns:
        if c.strip().lower() in name_candidates:
            return c
    # Otherwise: pick the last column if it's reasonably categorical or few unique values
    last = dataframe.columns[-1]
    # If last has too many unique numeric values, try to find a categorical-looking column
    if pd.api.types.is_numeric_dtype(dataframe[last]) and dataframe[last].nunique() > max(0.05*len(dataframe), 20):
        # pick a column with low cardinality
        low_card = [c for c in dataframe.columns if dataframe[c].nunique() <= max(0.05*len(dataframe), 20)]
        low_card = [c for c in low_card if c != last]
        if low_card:
            return low_card[-1]
    return last

y_col = TARGET_COLUMN or auto_pick_target(df)
if y_col not in df.columns:
    raise ValueError("Target column not found. Set TARGET_COLUMN to the correct label column.")

# ====== MAKE CLASSIFICATION TARGET IF NEEDED ======
y_raw = df[y_col]
# If numeric and high cardinality, discretize into 3 quantile bins (Low/Med/High)
if pd.api.types.is_numeric_dtype(y_raw) and y_raw.nunique() > max(0.05*len(df), 20):
    y = pd.qcut(y_raw, q=3, labels=["Low","Medium","High"])
else:
    y = y_raw.astype("category").astype(str)

X = df.drop(columns=[y_col])

# ====== FEATURE TYPES ======
numeric_features = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_features = [c for c in X.columns if c not in numeric_features]

# ====== PREPROCESSORS ======
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

# ====== TRAIN/TEST SPLIT ======
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# ====== MODEL ZOO ======
models = {
    "LogisticRegression": LogisticRegression(max_iter=200, n_jobs=None if hasattr(LogisticRegression(), "n_jobs") else None, class_weight="balanced"),
    "LinearSVC": LinearSVC(),
    "SVC_RBF": SVC(kernel="rbf", probability=True),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "AdaBoost": AdaBoostClassifier(random_state=RANDOM_STATE),
    "GaussianNB": GaussianNB(),
    "LinearDiscriminantAnalysis": LinearDiscriminantAnalysis(),
    "QuadraticDiscriminantAnalysis": QuadraticDiscriminantAnalysis(),
    "SGD_LogLoss": SGDClassifier(loss="log_loss", max_iter=1000, random_state=RANDOM_STATE)
}

# Some estimators (NB/LDA/QDA) need dense input; our OneHot is already dense.
# Build pipelines and evaluate
def fit_eval(name, clf):
    pipe = Pipeline(steps=[("prep", preprocessor), ("clf", clf)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print("="*80)
    print(f"{name} | Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds, digits=4))

for name, clf in models.items():
    try:
        fit_eval(name, clf)
    except Exception as e:
        print("="*80)
        print(f"{name} | SKIPPED due to error: {e}")

LogisticRegression | Accuracy: 0.5282
                  precision    recall  f1-score   support

IND-v0.1-modeled     0.9525    0.5312    0.6821      3810
             nan     0.0475    0.4684    0.0862       190

        accuracy                         0.5282      4000
       macro avg     0.5000    0.4998    0.3841      4000
    weighted avg     0.9095    0.5282    0.6538      4000

LinearSVC | Accuracy: 0.9525
                  precision    recall  f1-score   support

IND-v0.1-modeled     0.9525    1.0000    0.9757      3810
             nan     0.0000    0.0000    0.0000       190

        accuracy                         0.9525      4000
       macro avg     0.4763    0.5000    0.4878      4000
    weighted avg     0.9073    0.9525    0.9293      4000

SVC_RBF | Accuracy: 0.9525
                  precision    recall  f1-score   support

IND-v0.1-modeled     0.9525    1.0000    0.9757      3810
             nan     0.0000    0.0000    0.0000       190

        accuracy            

In [ ]:
# Unsupervised ML suite for India Pesticide Toxicity Risk dataset
# Algorithms: KMeans, Agglomerative, DBSCAN, GaussianMixture, Spectral, Birch
# Dimensionality Reduction: PCA, t-SNE (for visualization prep)

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Clustering algorithms
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering, Birch
from sklearn.mixture import GaussianMixture

# ====== CONFIG ======
DATA_PATHS = [
    "india_pesticide_toxicity_risk_reduced.csv"
]
N_CLUSTERS = 3   # Adjust depending on expected natural grouping
RANDOM_STATE = 42

# ====== LOAD ======
for p in DATA_PATHS:
    if os.path.exists(p):
        df = pd.read_csv(p)
        break
else:
    raise FileNotFoundError("Dataset not found.")

# ====== DROP LABEL IF EXISTS ======
# If dataset has a target label column, exclude it (we want unsupervised features only)
name_candidates = ["toxicity_risk","risk_label","label","target","class","toxicity","risk"]
target_col = None
for c in df.columns:
    if c.strip().lower() in name_candidates:
        target_col = c
        break
if target_col:
    X = df.drop(columns=[target_col])
else:
    X = df.copy()

# ====== FEATURE TYPES ======
numeric_features = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_features = [c for c in X.columns if c not in numeric_features]

# ====== PREPROCESSOR ======
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

# ====== PREPARE DATA ======
X_prepared = preprocessor.fit_transform(X)

# Convert sparse matrix to dense array for PCA and t-SNE
X_dense = X_prepared.toarray()

# ====== DIMENSIONALITY REDUCTION ======
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_dense)

# t-SNE for visualization prep (not needed for metrics, but for plots if required)
tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30)
X_tsne = tsne.fit_transform(X_dense)

# ====== CLUSTERING ALGORITHMS ======
clusterers = {
    "KMeans": KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE),
    "Agglomerative": AgglomerativeClustering(n_clusters=N_CLUSTERS),
    "DBSCAN": DBSCAN(eps=2.0, min_samples=5),
    "GaussianMixture": GaussianMixture(n_components=N_CLUSTERS, random_state=RANDOM_STATE),
    "SpectralClustering": SpectralClustering(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, affinity="nearest_neighbors"),
    "Birch": Birch(n_clusters=N_CLUSTERS)
}

# ====== EVALUATION ======
def evaluate_clustering(name, model, X_data):
    try:
        if hasattr(model, "fit_predict"):
            labels = model.fit_predict(X_data)
        else:
            model.fit(X_data)
            labels = model.predict(X_data)

        # Some algorithms may assign all to one cluster (silhouette fails)
        if len(set(labels)) > 1 and -1 not in set(labels):
            sil = silhouette_score(X_data, labels)
            dbi = davies_bouldin_score(X_data, labels)
        else:
            sil, dbi = None, None

        print("="*80)
        print(f"{name}")
        print(f"Cluster labels: {np.unique(labels)}")
        if sil is not None:
            print(f"Silhouette Score: {sil:.4f}")
            print(f"Davies-Bouldin Index: {dbi:.4f}")
        if name == "KMeans":
            print(f"Inertia: {model.inertia_:.4f}")
    except Exception as e:
        print("="*80)
        print(f"{name} | SKIPPED due to error: {e}")

# Run all
for name, model in clusterers.items():
    # Note: Some clustering algorithms (like SpectralClustering, GaussianMixture)
    # may perform better or require dense input.
    # We will use X_dense for all for simplicity, but be aware of this.
    evaluate_clustering(name, model, X_dense)